# Chapter 18 &mdash; Church Numerals: Numbers as Repeated Application

**Concept 4 of the Chapter 18 decomposition:** *Church Numerals: Numbers as Repeated Application*

$n$ is the function that applies its argument $n$ times; `ZERO`, `SUCC`, `ADD` and `MUL` follow.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Church-Numerals/Concept-Church-Numerals.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


There are no numbers in the $\lambda$-calculus, so we **encode** them.

$$\overline{n} = \lambda f.\lambda x.\,f^n(x)$$

$\overline{0}$ applies $f$ zero times; $\overline{3}$ applies it three times. A
numeral **is** a repetition operator.

The arithmetic follows from that reading:

* $\textbf{SUCC} = \lambda n.\lambda f.\lambda x.\,f\,(n\,f\,x)$ &mdash; apply once more;
* $\textbf{ADD} = \lambda m.\lambda n.\lambda f.\lambda x.\,m\,f\,(n\,f\,x)$ &mdash; do
  $n$ applications, then $m$ more;
* $\textbf{MUL} = \lambda m.\lambda n.\lambda f.\,m\,(n\,f)$ &mdash; $m$ repetitions of
  "$n$ repetitions";
* $\textbf{EXP} = \lambda m.\lambda n.\,n\,m$ &mdash; startlingly.

**Predecessor** is the hard one, and it needs pairs (Concept 5).

## 2. Definitions

### The encoding

In [ ]:
# --- Church encodings, in Python lambdas --------------------------------
# A Church numeral n is the function that applies f to x, n times.
ZERO  = lambda f: lambda x: x
SUCC  = lambda n: lambda f: lambda x: f(n(f)(x))
ADD   = lambda m: lambda n: lambda f: lambda x: m(f)(n(f)(x))
MUL   = lambda m: lambda n: lambda f: m(n(f))
EXP   = lambda m: lambda n: n(m)

def church(n):
    c = ZERO
    for _ in range(n): c = SUCC(c)
    return c

def unchurch(c):
    return c(lambda k: k + 1)(0)

# Booleans: TRUE picks its first argument, FALSE its second -- so a
# Boolean IS an if-then-else.
TRUE  = lambda t: lambda f: t
FALSE = lambda t: lambda f: f
IF    = lambda b: lambda t: lambda f: b(t)(f)
AND   = lambda p: lambda q: p(q)(p)
OR    = lambda p: lambda q: p(p)(q)
NOT   = lambda p: p(FALSE)(TRUE)

def unbool(b): return b(True)(False)

# Pairs and selectors
PAIR   = lambda a: lambda b: lambda s: s(a)(b)
FIRST  = lambda p: p(TRUE)
SECOND = lambda p: p(FALSE)

# Predecessor and zero-test, which recursion needs
ISZERO = lambda n: n(lambda _: FALSE)(TRUE)
SHIFT  = lambda p: PAIR(SECOND(p))(SUCC(SECOND(p)))
PRED   = lambda n: FIRST(n(SHIFT)(PAIR(ZERO)(ZERO)))
SUB    = lambda m: lambda n: n(PRED)(m)
LEQ    = lambda m: lambda n: ISZERO(SUB(m)(n))

### Seeing the repetition

In [ ]:
def trace_numeral(n):
    # prepend 'f(' n times, then close the parentheses
    return church(n)(lambda s: "f(" + s)("x") + ")" * n

## 3. Tests

A numeral literally repeats its argument.

In [ ]:
for n in range(5):
    print("  church(%d) applied to 'f' and 'x' : %s" % (n, trace_numeral(n)))
assert trace_numeral(3) == "f(f(f(x)))"

And `unchurch` just counts the applications.

In [ ]:
for n in range(6):
    assert unchurch(church(n)) == n
print("round-trip verified for 0..5")
print("  church(4)(lambda k: k+1)(0) =", church(4)(lambda k: k + 1)(0))

`SUCC`, `ADD`, `MUL`, `EXP`.

In [ ]:
print("  SUCC 4      =", unchurch(SUCC(church(4))))
print("  ADD 3 4     =", unchurch(ADD(church(3))(church(4))))
print("  MUL 3 4     =", unchurch(MUL(church(3))(church(4))))
print("  EXP 2 5     =", unchurch(EXP(church(2))(church(5))))
assert unchurch(SUCC(church(4))) == 5
assert unchurch(ADD(church(3))(church(4))) == 7
assert unchurch(MUL(church(3))(church(4))) == 12
assert unchurch(EXP(church(2))(church(5))) == 32

Exhaustively, on a small range.

In [ ]:
for m in range(5):
    for n in range(5):
        assert unchurch(ADD(church(m))(church(n))) == m + n
        assert unchurch(MUL(church(m))(church(n))) == m * n
        if n <= 3 and m <= 3:
            assert unchurch(EXP(church(m))(church(n))) == m ** n
print("ADD and MUL for all m,n in 0..4; EXP for 0..3 -- all correct")

`EXP m n = n m` &mdash; why that works.

In [ ]:
print("n is 'apply its argument n times'")
print("so n applied to m is 'apply m, n times'")
print("and applying the function 'multiply by m' n times to 1 is m^n")
print()
print("  EXP 3 2 = 2 3 =", unchurch(EXP(church(3))(church(2))))
assert unchurch(EXP(church(3))(church(2))) == 9

Multiplication is composition, which the code shows plainly.

In [ ]:
double = lambda g: lambda x: g(g(x))
print("  MUL m n = lambda f: m(n(f))  -- m repetitions of (n repetitions of f)")
comp = MUL(church(3))(church(2))
print("  applied to +1 from 0 :", comp(lambda k: k + 1)(0))
assert comp(lambda k: k + 1)(0) == 6

## 4. Exercises


1. Define `DOUBLE` two ways: with `ADD` and with `MUL`. Are the terms the same?
2. What is $\overline{0}$ applied to anything? What does that say about `EXP m 0`?
3. Why is predecessor hard when successor is trivial?

In [ ]:
# Your work for the exercises above.